# Closed-loop adaptive GRAPE

This notebook keeps the same simplified style as `test_grape.ipynb` and `adaptive_grape_parameter_fit.ipynb`, but now combines the pieces into a small closed-loop calibration experiment.

The loop is:

1. optimize a pulse with GRAPE using the current calibrated model,
2. probe nearby pulses on the hidden true model with 500-shot binary measurements,
3. fit Hamiltonian parameters and the measurement response `A, B` from all measured data,
4. repeat, always warm-starting from the previous optimized pulse and previous fitted parameters.

The first phase uses unitary evolution for GRAPE and fitting. The second phase continues from that result and uses non-unitary evolution for GRAPE and fitting, while still keeping `T1/T2` fixed rather than fitting them.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

cwd = Path.cwd()
project_dir = cwd if (cwd / "grape.py").exists() else cwd / "Simplified_adaptive_grape"
repo_root = project_dir.parent
sys.path.insert(0, str(project_dir))
sys.path.insert(0, str(repo_root))

import toolbox as tbx
import grape


## Experimental values and hidden true model

The nominal model is what GRAPE believes. The hidden true model is what generates the fake experimental measurements. It is intentionally close to the nominal model, but has small frequency shifts, amplitude miscalibrations, and slightly shorter coherence times.

The measured value is not the true photon probability directly. We first sample a 500-shot binomial distribution with probability `P_n`, then apply `observed = A*x + B`, with `0 -> 0.01` and `1 -> 0.93`, to mimic the long selective pulse plus readout imperfections.


In [ ]:
config_path = repo_root / "configuration.json"
with config_path.open("r") as f:
    cfg = json.load(f)

# kHz -> MHz -> rad/us. We use a negative chi so H_disp = chi * n_cavity * |e><e|.
chi_nominal = -2 * jnp.pi * cfg["chi_kHz"] * 1e-3
cavity_self_kerr_nominal = 2 * jnp.pi * cfg["self_Kerr_kHz"] * 1e-3

qubit_T1_nominal = float(cfg["qubit_T1_us"])
qubit_T2_nominal = float(cfg["qubit_T2_us"])
cavity_T1_nominal = float(cfg["storage_T1_us"])
cavity_T2_nominal = float(cfg["storage_T2_us"])

mu_qub = 20.0
mu_cav = 20.0

# Hidden true model. Same soft mismatch as the parameter-fit notebook.
chi_true_shift_MHz = 0.001
qubit_freq_shift_MHz = 0.020
cavity_freq_shift_MHz = 0.040
cavity_self_kerr_true_MHz = -0.0007
qubit_amp_factor_true = 1.0005
cavity_amp_factor_true = 0.995

chi_true = chi_nominal + 2 * jnp.pi * chi_true_shift_MHz
qubit_shift_true = 2 * jnp.pi * qubit_freq_shift_MHz
cavity_shift_true = 2 * jnp.pi * cavity_freq_shift_MHz
cavity_self_kerr_true = 2 * jnp.pi * cavity_self_kerr_true_MHz

qubit_T1_true = 0.96 * qubit_T1_nominal
qubit_T2_true = 0.93 * qubit_T2_nominal
cavity_T1_true = 0.98 * cavity_T1_nominal
cavity_T2_true = 0.96 * cavity_T2_nominal

nominal_model = {
    "chi": chi_nominal,
    "qubit_shift": 0.0,
    "cavity_shift": 0.0,
    "cavity_self_kerr": cavity_self_kerr_nominal,
    "qubit_amp_factor": 1.0,
    "cavity_amp_factor": 1.0,
    "qubit_T1": qubit_T1_nominal,
    "qubit_T2": qubit_T2_nominal,
    "cavity_T1": cavity_T1_nominal,
    "cavity_T2": cavity_T2_nominal,
}

true_model = {
    "chi": chi_true,
    "qubit_shift": qubit_shift_true,
    "cavity_shift": cavity_shift_true,
    "cavity_self_kerr": cavity_self_kerr_true,
    "qubit_amp_factor": qubit_amp_factor_true,
    "cavity_amp_factor": cavity_amp_factor_true,
    "qubit_T1": qubit_T1_true,
    "qubit_T2": qubit_T2_true,
    "cavity_T1": cavity_T1_true,
    "cavity_T2": cavity_T2_true,
}

shots_per_pulse = 500
A_true = 0.92
B_true = 0.01

print("nominal chi [rad/us]:", float(chi_nominal))
print("true chi shift [MHz]:", chi_true_shift_MHz)
print("true qubit/cavity shifts [MHz]:", qubit_freq_shift_MHz, cavity_freq_shift_MHz)
print("true A, B:", A_true, B_true)


## Pulse basis and simulation grid

The control is still `4 x 20` real B-spline coefficients. The two skipped splines at each end force every basis pulse to start and end at zero. With quadratic splines, at most three basis functions overlap at a given time.


In [ ]:
N_cav = 25
target_n = 2

param_clip = 2.0
n_channels = 4
n_total_bsplines = 24
spline_degree = 2
skip_left = 2
skip_right = 2
bspln_num = n_total_bsplines - skip_left - skip_right
assert bspln_num == 20

T_us = 1.408
Nt = 130

time_start = 0.0
time_end = T_us
time_edges = jnp.linspace(time_start, time_end, Nt + 1)
time_mids = 0.5 * (time_edges[1:] + time_edges[:-1])
time_intervals = time_edges[1:] - time_edges[:-1]

bspline_builder = tbx.setup_bspline_builder(
    time_start,
    time_end,
    n_total_bsplines,
    spline_degree,
    skip_left,
    skip_right,
)
bsplns_mids = jnp.asarray(bspline_builder(np.asarray(time_mids)))
bsplns_edges = jnp.asarray(bspline_builder(np.asarray(time_edges)))

print("B-splines on midpoints:", bsplns_mids.shape)
print("endpoint max:", float(jnp.max(jnp.abs(bsplns_edges[:, [0, -1]]))))
print("max active B-splines:", int(jnp.max(jnp.sum(bsplns_mids > 1e-12, axis=0))))

plt.figure(figsize=(9, 3))
for b in np.asarray(bsplns_edges):
    plt.plot(np.asarray(time_edges) * 1e3, b, lw=1)
plt.xlabel("time [ns]")
plt.ylabel("basis value")
plt.title("20 quadratic B-splines after skipping two at each edge")
plt.grid(alpha=0.3)
plt.show()


## Operators and simulators


In [ ]:
a = tbx.tensor(tbx.identity(2), tbx.destroy(N_cav))
adag = tbx.hconj(a)
n_phot = adag @ a
n2_minus_n = n_phot @ n_phot - n_phot

sigz = tbx.tensor(tbx.sigma.z, tbx.identity(N_cav))
sigp = tbx.tensor(tbx.sigma.p, tbx.identity(N_cav))  # |e> -> |g>
sigm = tbx.hconj(sigp)                              # |g> -> |e>
one = tbx.identity(2 * N_cav)
qubit_excited = 0.5 * (one - sigz)

psi_init = tbx.tensor(tbx.basis(2, 0), tbx.basis(N_cav, 0))
rho_init = psi_init @ tbx.hconj(psi_init)

print("Hilbert dimension:", psi_init.shape[0])


In [ ]:
def hamiltonian_tree(ctrl_coeffs, model):
    e_qub, e_cav = grape.controls_from_coefficients(ctrl_coeffs, bsplns_mids)
    e_qub = model["qubit_amp_factor"] * e_qub
    e_cav = model["cavity_amp_factor"] * 1j * jnp.conj(e_cav)

    H_drift = (
        model["chi"] * (n_phot @ qubit_excited)
        + 0.5 * model["cavity_self_kerr"] * n2_minus_n
        + model["cavity_shift"] * n_phot
        + model["qubit_shift"] * qubit_excited
    )

    return [
        [H_drift, 1.0, 1.0, 0.0],
        [sigp, mu_qub * e_qub, 1.0, 1.0],
        [adag, mu_cav * e_cav, 1.0, 1.0],
    ]


def collapse_ops(model):
    gamma_phi_qub = jnp.maximum(1.0 / model["qubit_T2"] - 0.5 / model["qubit_T1"], 0.0)
    gamma_phi_cav = jnp.maximum(1.0 / model["cavity_T2"] - 0.5 / model["cavity_T1"], 0.0)

    return [
        jnp.sqrt(1.0 / model["qubit_T1"]) * sigp,
        jnp.sqrt(2.0 * gamma_phi_qub) * qubit_excited,
        jnp.sqrt(1.0 / model["cavity_T1"]) * a,
        jnp.sqrt(2.0 * gamma_phi_cav) * n_phot,
    ]


def unitary_probability(ctrl_coeffs, model):
    psi_t = tbx.sesolve_htree(hamiltonian_tree(ctrl_coeffs, model), psi_init, time_intervals)
    return grape.fock_probability_from_state(psi_t[-1], N_cav, target_n)


def decay_probability(ctrl_coeffs, model):
    rho_final = tbx.mesolve_htree(
        hamiltonian_tree(ctrl_coeffs, model),
        collapse_ops(model),
        rho_init,
        time_intervals,
    )
    return grape.fock_probability_from_density(rho_final, N_cav, target_n)


## True experimental measurement

The true experiment always uses the hidden non-unitary model. The calibration algorithm never sees `p_true`; it only sees the noisy observed value.


In [ ]:
@jax.jit
def true_physical_probability(ctrl_coeffs):
    return decay_probability(ctrl_coeffs, true_model)


def measure_true_experiment(ctrl_coeffs, key):
    p_true = true_physical_probability(ctrl_coeffs)
    successes = jax.random.binomial(key, n=shots_per_pulse, p=jnp.clip(p_true, 0.0, 1.0))
    shot_fraction = successes / shots_per_pulse
    observed = A_true * shot_fraction + B_true
    return observed, successes, p_true


## Calibrated model parameters

We fit only Hamiltonian and readout-response parameters:

- chi shift,
- qubit frequency shift,
- cavity self-Kerr,
- cavity frequency shift,
- qubit amplitude factor,
- cavity amplitude factor,
- `A` and `B` in the measurement response.

We do **not** fit `T1/T2`; those stay fixed at the nominal values. In the unitary phase they are unused. In the non-unitary phase they are included in the simulator but not adjusted.


In [ ]:
def model_from_fit_raw(raw):
    chi_shift_MHz = 0.010 * jnp.tanh(raw[0])
    qubit_shift_MHz = 0.080 * jnp.tanh(raw[1])
    cavity_self_kerr_MHz = cfg["self_Kerr_kHz"] * 1e-3 + 0.0015 * jnp.tanh(raw[2])
    cavity_shift_MHz = 0.120 * jnp.tanh(raw[3])

    qubit_amp_factor = 1.0 + 0.010 * jnp.tanh(raw[4])
    cavity_amp_factor = 1.0 + 0.030 * jnp.tanh(raw[5])

    A_fit = 0.90 + 0.08 * jnp.tanh(raw[6])
    B_fit = 0.02 + 0.03 * jnp.tanh(raw[7])

    model = {
        "chi": chi_nominal + 2 * jnp.pi * chi_shift_MHz,
        "qubit_shift": 2 * jnp.pi * qubit_shift_MHz,
        "cavity_shift": 2 * jnp.pi * cavity_shift_MHz,
        "cavity_self_kerr": 2 * jnp.pi * cavity_self_kerr_MHz,
        "qubit_amp_factor": qubit_amp_factor,
        "cavity_amp_factor": cavity_amp_factor,
        "qubit_T1": qubit_T1_nominal,
        "qubit_T2": qubit_T2_nominal,
        "cavity_T1": cavity_T1_nominal,
        "cavity_T2": cavity_T2_nominal,
    }
    response = {"A": A_fit, "B": B_fit}
    return model, response


def predicted_observed_unitary(raw, ctrl_coeffs):
    model, response = model_from_fit_raw(raw)
    p_model = unitary_probability(ctrl_coeffs, model)
    return response["A"] * p_model + response["B"]


def predicted_observed_decay(raw, ctrl_coeffs):
    model, response = model_from_fit_raw(raw)
    p_model = decay_probability(ctrl_coeffs, model)
    return response["A"] * p_model + response["B"]


def parameter_summary(raw):
    model, response = model_from_fit_raw(raw)
    return {
        "chi_shift_MHz": float((model["chi"] - chi_nominal) / (2 * jnp.pi)),
        "qubit_shift_MHz": float(model["qubit_shift"] / (2 * jnp.pi)),
        "cavity_shift_MHz": float(model["cavity_shift"] / (2 * jnp.pi)),
        "self_kerr_MHz": float(model["cavity_self_kerr"] / (2 * jnp.pi)),
        "qubit_amp": float(model["qubit_amp_factor"]),
        "cavity_amp": float(model["cavity_amp_factor"]),
        "A": float(response["A"]),
        "B": float(response["B"]),
    }


## Local probing around the optimized pulse

Most pulses are extremely close to the GRAPE optimum; a smaller fraction are slightly farther away. This gives the fit local information without spending most shots on obviously bad pulses.


In [ ]:
dataset_size_per_round = 500
very_local_noise_std = 0.0025
medium_local_noise_std = 0.0060
very_local_fraction = 0.90


def probe_local_dataset(ctrl_center, key, label):
    center_true = true_physical_probability(ctrl_center)
    center_expected_observed = A_true * center_true + B_true
    print(f"{label} center hidden true P_n: {float(center_true):.6f}")
    print(f"{label} center expected observed: {float(center_expected_observed):.6f}")

    n_very_local = int(dataset_size_per_round * very_local_fraction)
    noise_scales = jnp.concatenate([
        very_local_noise_std * jnp.ones((n_very_local,)),
        medium_local_noise_std * jnp.ones((dataset_size_per_round - n_very_local,)),
    ])

    key, noise_key, perm_key = jax.random.split(key, 3)
    noise_scales = noise_scales[jax.random.permutation(perm_key, dataset_size_per_round)]
    noise = noise_scales[:, None, None] * jax.random.normal(
        noise_key,
        (dataset_size_per_round, n_channels, bspln_num),
    )

    controls = grape.clip_coefficients(ctrl_center[None, :, :] + noise, param_clip)
    controls = controls.at[0].set(ctrl_center)

    observed_values = []
    true_values = []
    successes_values = []

    pbar = tqdm(range(dataset_size_per_round), desc=f"probe {label}", leave=False)
    for i in pbar:
        key, measure_key = jax.random.split(key)
        observed, successes, p_true = measure_true_experiment(controls[i], measure_key)
        observed_values.append(observed)
        true_values.append(p_true)
        successes_values.append(successes)
        pbar.set_postfix(obs=f"{float(observed):.4f}", true=f"{float(p_true):.4f}")

    observed_values = jnp.asarray(observed_values)
    true_values = jnp.asarray(true_values)
    successes_values = jnp.asarray(successes_values)

    print(f"{label} observed mean/std: {float(jnp.mean(observed_values)):.5f} / {float(jnp.std(observed_values)):.5f}")
    print(f"{label} true P_n mean/best: {float(jnp.mean(true_values)):.5f} / {float(jnp.max(true_values)):.5f}")
    return key, controls, observed_values, successes_values, true_values


def append_dataset(old_controls, old_observed, old_true, new_controls, new_observed, new_true):
    if old_controls is None:
        return new_controls, new_observed, new_true
    return (
        jnp.concatenate([old_controls, new_controls], axis=0),
        jnp.concatenate([old_observed, new_observed], axis=0),
        jnp.concatenate([old_true, new_true], axis=0),
    )


## GRAPE steps

The GRAPE optimizer only uses the calibrated model, never the hidden true model. The hidden true model is used afterward to create measurements.


In [ ]:
unitary_grape_steps_per_round = 250
unitary_grape_learning_rate = 0.020
unitary_grape_optimizer = optax.adam(unitary_grape_learning_rate)

nonunitary_grape_steps_per_round = 140
nonunitary_grape_learning_rate = 0.002
nonunitary_grape_optimizer = optax.adam(nonunitary_grape_learning_rate)


@jax.jit
def unitary_grape_step(ctrl_coeffs, opt_state, fit_raw):
    def loss_fn(c):
        model, _ = model_from_fit_raw(fit_raw)
        p = unitary_probability(c, model)
        return 1.0 - p + grape.pulse_penalty(c), p

    (loss_value, p_value), grads = jax.value_and_grad(loss_fn, has_aux=True)(ctrl_coeffs)
    updates, opt_state = unitary_grape_optimizer.update(grads, opt_state)
    ctrl_coeffs = optax.apply_updates(ctrl_coeffs, updates)
    ctrl_coeffs = grape.clip_coefficients(ctrl_coeffs, param_clip)
    return ctrl_coeffs, opt_state, loss_value, p_value


@jax.jit
def nonunitary_grape_step(ctrl_coeffs, opt_state, fit_raw):
    def loss_fn(c):
        model, _ = model_from_fit_raw(fit_raw)
        p = decay_probability(c, model)
        return 1.0 - p + grape.pulse_penalty(c), p

    (loss_value, p_value), grads = jax.value_and_grad(loss_fn, has_aux=True)(ctrl_coeffs)
    updates, opt_state = nonunitary_grape_optimizer.update(grads, opt_state)
    ctrl_coeffs = optax.apply_updates(ctrl_coeffs, updates)
    ctrl_coeffs = grape.clip_coefficients(ctrl_coeffs, param_clip)
    return ctrl_coeffs, opt_state, loss_value, p_value


def run_unitary_grape(ctrl_start, fit_raw, label):
    ctrl = ctrl_start
    opt_state = unitary_grape_optimizer.init(ctrl)
    history = []
    pbar = tqdm(range(unitary_grape_steps_per_round), desc=f"unitary GRAPE {label}", leave=False)
    for _ in pbar:
        ctrl, opt_state, loss_value, p_value = unitary_grape_step(ctrl, opt_state, fit_raw)
        history.append(float(p_value))
        pbar.set_postfix(P_n=f"{float(p_value):.5f}")
    return ctrl, history


def run_nonunitary_grape(ctrl_start, fit_raw, label):
    ctrl = ctrl_start
    opt_state = nonunitary_grape_optimizer.init(ctrl)
    history = []
    pbar = tqdm(range(nonunitary_grape_steps_per_round), desc=f"nonunitary GRAPE {label}", leave=False)
    for _ in pbar:
        ctrl, opt_state, loss_value, p_value = nonunitary_grape_step(ctrl, opt_state, fit_raw)
        history.append(float(p_value))
        pbar.set_postfix(P_n=f"{float(p_value):.5f}")
    return ctrl, history


## Parameter fitting steps

The fit sees only measured values. It minimizes

`mean((predicted_observed(control) - measured_observed(control))**2)`.

The unitary phase uses `predicted_observed_unitary`; the non-unitary phase uses `predicted_observed_decay`. In both cases, the parameter vector is initialized from the previous round.


In [ ]:
fit_steps_per_round = 180
fit_batch_size = 8
fit_learning_rate = 0.020
fit_optimizer = optax.adam(fit_learning_rate)


@jax.jit
def fit_step_unitary(fit_raw, opt_state, batch_controls, batch_observed):
    def loss_fn(raw):
        pred = jax.vmap(lambda c: predicted_observed_unitary(raw, c))(batch_controls)
        return jnp.mean((pred - batch_observed) ** 2), pred

    (loss_value, pred), grads = jax.value_and_grad(loss_fn, has_aux=True)(fit_raw)
    updates, opt_state = fit_optimizer.update(grads, opt_state)
    fit_raw = optax.apply_updates(fit_raw, updates)
    return fit_raw, opt_state, loss_value, jnp.mean(pred)


@jax.jit
def fit_step_decay(fit_raw, opt_state, batch_controls, batch_observed):
    def loss_fn(raw):
        pred = jax.vmap(lambda c: predicted_observed_decay(raw, c))(batch_controls)
        return jnp.mean((pred - batch_observed) ** 2), pred

    (loss_value, pred), grads = jax.value_and_grad(loss_fn, has_aux=True)(fit_raw)
    updates, opt_state = fit_optimizer.update(grads, opt_state)
    fit_raw = optax.apply_updates(fit_raw, updates)
    return fit_raw, opt_state, loss_value, jnp.mean(pred)


def fit_parameters(fit_raw, controls_seen, observed_seen, key, mode, label):
    opt_state = fit_optimizer.init(fit_raw)
    losses = []
    n_seen = controls_seen.shape[0]

    pbar = tqdm(range(fit_steps_per_round), desc=f"fit {mode} model {label}", leave=False)
    for _ in pbar:
        key, batch_key = jax.random.split(key)
        batch_idx = jax.random.choice(batch_key, n_seen, shape=(fit_batch_size,), replace=False)
        if mode == "unitary":
            fit_raw, opt_state, loss_value, mean_pred = fit_step_unitary(
                fit_raw,
                opt_state,
                controls_seen[batch_idx],
                observed_seen[batch_idx],
            )
        else:
            fit_raw, opt_state, loss_value, mean_pred = fit_step_decay(
                fit_raw,
                opt_state,
                controls_seen[batch_idx],
                observed_seen[batch_idx],
            )
        losses.append(float(loss_value))
        pbar.set_postfix(loss=f"{float(loss_value):.3e}", pred=f"{float(mean_pred):.4f}")

    print(f"{label} fit loss first/last: {losses[0]:.4e} / {losses[-1]:.4e}")
    print(f"{label} fitted parameters:", parameter_summary(fit_raw))
    return key, fit_raw, losses


## Initial pulse and shared state


In [ ]:
key = jax.random.key(123)
key, subkey = jax.random.split(key)

initial_coeffs = 0.03 * jax.random.normal(subkey, (n_channels, bspln_num))
initial_coeffs = grape.clip_coefficients(initial_coeffs, param_clip)

fit_raw = jnp.zeros((8,))
controls_seen = None
observed_seen = None
true_seen = None

print("initial true P_n:", float(true_physical_probability(initial_coeffs)))
print("initial fitted parameters:", parameter_summary(fit_raw))


## Phase 1: closed-loop unitary adaptive GRAPE

Each round runs unitary GRAPE, measures local pulses on the hidden true open-system model, then refits Hamiltonian and measurement-response parameters using unitary evolution.


In [ ]:
n_unitary_rounds = 4
ctrl_current = initial_coeffs
unitary_records = []
unitary_grape_histories = []
unitary_fit_histories = []

for round_index in range(n_unitary_rounds):
    label = f"unitary round {round_index:02d}"
    print("\n" + "=" * 80)
    print(label)
    print("starting parameters:", parameter_summary(fit_raw))

    ctrl_current, grape_history = run_unitary_grape(ctrl_current, fit_raw, label)
    unitary_grape_histories.append(grape_history)

    fitted_model, fitted_response = model_from_fit_raw(fit_raw)
    predicted_p = unitary_probability(ctrl_current, fitted_model)
    hidden_true_p = true_physical_probability(ctrl_current)
    expected_observed = A_true * hidden_true_p + B_true
    print(f"{label} GRAPE predicted unitary P_n: {float(predicted_p):.6f}")
    print(f"{label} hidden true P_n: {float(hidden_true_p):.6f}")
    print(f"{label} expected observed value: {float(expected_observed):.6f}")

    key, new_controls, new_observed, new_successes, new_true = probe_local_dataset(ctrl_current, key, label)
    controls_seen, observed_seen, true_seen = append_dataset(
        controls_seen,
        observed_seen,
        true_seen,
        new_controls,
        new_observed,
        new_true,
    )
    print(f"total measured controls: {controls_seen.shape[0]}")

    key, fit_raw, fit_losses = fit_parameters(
        fit_raw,
        controls_seen,
        observed_seen,
        key,
        mode="unitary",
        label=label,
    )
    unitary_fit_histories.append(fit_losses)

    unitary_records.append({
        "round": round_index,
        "predicted_p": float(predicted_p),
        "true_p": float(hidden_true_p),
        "expected_observed": float(expected_observed),
        "data_observed_mean": float(jnp.mean(new_observed)),
        "data_true_mean": float(jnp.mean(new_true)),
        "fit_loss_last": float(fit_losses[-1]),
    })


## Phase 2: closed-loop non-unitary adaptive GRAPE

This phase starts from the unitary result. It uses non-unitary GRAPE and non-unitary parameter fitting, but the fitted parameters are still only Hamiltonian and measurement-response parameters. The nominal `T1/T2` values are used but not fitted.


In [ ]:
n_nonunitary_rounds = 3
nonunitary_records = []
nonunitary_grape_histories = []
nonunitary_fit_histories = []

for round_index in range(n_nonunitary_rounds):
    label = f"nonunitary round {round_index:02d}"
    print("\n" + "=" * 80)
    print(label)
    print("starting parameters:", parameter_summary(fit_raw))

    ctrl_current, grape_history = run_nonunitary_grape(ctrl_current, fit_raw, label)
    nonunitary_grape_histories.append(grape_history)

    fitted_model, fitted_response = model_from_fit_raw(fit_raw)
    predicted_p = decay_probability(ctrl_current, fitted_model)
    hidden_true_p = true_physical_probability(ctrl_current)
    expected_observed = A_true * hidden_true_p + B_true
    print(f"{label} GRAPE predicted decay P_n: {float(predicted_p):.6f}")
    print(f"{label} hidden true P_n: {float(hidden_true_p):.6f}")
    print(f"{label} expected observed value: {float(expected_observed):.6f}")

    key, new_controls, new_observed, new_successes, new_true = probe_local_dataset(ctrl_current, key, label)
    controls_seen, observed_seen, true_seen = append_dataset(
        controls_seen,
        observed_seen,
        true_seen,
        new_controls,
        new_observed,
        new_true,
    )
    print(f"total measured controls: {controls_seen.shape[0]}")

    key, fit_raw, fit_losses = fit_parameters(
        fit_raw,
        controls_seen,
        observed_seen,
        key,
        mode="decay",
        label=label,
    )
    nonunitary_fit_histories.append(fit_losses)

    nonunitary_records.append({
        "round": round_index,
        "predicted_p": float(predicted_p),
        "true_p": float(hidden_true_p),
        "expected_observed": float(expected_observed),
        "data_observed_mean": float(jnp.mean(new_observed)),
        "data_true_mean": float(jnp.mean(new_true)),
        "fit_loss_last": float(fit_losses[-1]),
    })


## Diagnostics


In [ ]:
def plot_records(records, title):
    rounds = np.arange(len(records))
    plt.plot(rounds, [r["true_p"] for r in records], "o-", label="hidden true P_n of GRAPE pulse")
    plt.plot(rounds, [r["predicted_p"] for r in records], "o-", label="model predicted P_n")
    plt.plot(rounds, [r["data_true_mean"] for r in records], "o-", label="mean true P_n in local data")
    plt.xlabel("adaptive round")
    plt.ylabel(f"P_{target_n}")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend()

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plot_records(unitary_records, "unitary phase")
plt.subplot(1, 2, 2)
plot_records(nonunitary_records, "non-unitary phase")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 3))
all_true = [r["true_p"] for r in unitary_records] + [r["true_p"] for r in nonunitary_records]
all_labels = [f"U{i}" for i in range(len(unitary_records))] + [f"D{i}" for i in range(len(nonunitary_records))]
plt.plot(all_true, "o-")
plt.xticks(range(len(all_labels)), all_labels)
plt.ylabel(f"hidden true P_{target_n}")
plt.title("closed-loop optimized pulse quality")
plt.grid(alpha=0.3)
plt.show()

print("final fitted parameters:", parameter_summary(fit_raw))
print("final hidden true P_n:", float(true_physical_probability(ctrl_current)))
print("final expected observed value:", float(A_true * true_physical_probability(ctrl_current) + B_true))
